In [2]:
import gc

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application")
#auc_score_OOF= 0.754

#freeing memory
del merged_df
gc.collect()



[0]	validation_0-auc:0.71367
[1]	validation_0-auc:0.72475
[2]	validation_0-auc:0.72876
[3]	validation_0-auc:0.73188
[4]	validation_0-auc:0.73575
[5]	validation_0-auc:0.73836
[6]	validation_0-auc:0.74116
[7]	validation_0-auc:0.74387
[8]	validation_0-auc:0.74634
[9]	validation_0-auc:0.74739
[10]	validation_0-auc:0.74941
[11]	validation_0-auc:0.75088
[12]	validation_0-auc:0.75159
[13]	validation_0-auc:0.75233
[14]	validation_0-auc:0.75353
[15]	validation_0-auc:0.75431
[16]	validation_0-auc:0.75581
[17]	validation_0-auc:0.75612
[18]	validation_0-auc:0.75685
[19]	validation_0-auc:0.75725
[20]	validation_0-auc:0.75754
[21]	validation_0-auc:0.75786
[22]	validation_0-auc:0.75828
[23]	validation_0-auc:0.75863
[24]	validation_0-auc:0.75856
[25]	validation_0-auc:0.75874
[26]	validation_0-auc:0.75878
[27]	validation_0-auc:0.75855
[28]	validation_0-auc:0.75892
[29]	validation_0-auc:0.75900
[30]	validation_0-auc:0.75892
[31]	validation_0-auc:0.75897
[32]	validation_0-auc:0.75927
[33]	validation_0-au

442

In [ ]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau")
#auc_score_OOF= 0.752

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71408
[1]	validation_0-auc:0.72260
[2]	validation_0-auc:0.72744
[3]	validation_0-auc:0.72960
[4]	validation_0-auc:0.73170
[5]	validation_0-auc:0.73427
[6]	validation_0-auc:0.73735
[7]	validation_0-auc:0.73969
[8]	validation_0-auc:0.74285
[9]	validation_0-auc:0.74424
[10]	validation_0-auc:0.74548
[11]	validation_0-auc:0.74787
[12]	validation_0-auc:0.74857
[13]	validation_0-auc:0.74952
[14]	validation_0-auc:0.75146
[15]	validation_0-auc:0.75238
[16]	validation_0-auc:0.75305
[17]	validation_0-auc:0.75345
[18]	validation_0-auc:0.75397
[19]	validation_0-auc:0.75441
[20]	validation_0-auc:0.75459
[21]	validation_0-auc:0.75521
[22]	validation_0-auc:0.75525
[23]	validation_0-auc:0.75543
[24]	validation_0-auc:0.75561
[25]	validation_0-auc:0.75540
[26]	validation_0-auc:0.75569
[27]	validation_0-auc:0.75624
[28]	validation_0-auc:0.75641
[29]	validation_0-auc:0.75631
[30]	validation_0-auc:0.75622
[31]	validation_0-auc:0.75615
[32]	validation_0-auc:0.75588
[33]	validation_0-au

691

In [ ]:
#finally we try with app_train + prev_app + bureau
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.0 (app_train+bureau+prev_app)")
#auc_score_OOF= 0.760 is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71372
[1]	validation_0-auc:0.72447
[2]	validation_0-auc:0.72929
[3]	validation_0-auc:0.73221
[4]	validation_0-auc:0.73550
[5]	validation_0-auc:0.73839
[6]	validation_0-auc:0.74131
[7]	validation_0-auc:0.74404
[8]	validation_0-auc:0.74652
[9]	validation_0-auc:0.75023
[10]	validation_0-auc:0.75145
[11]	validation_0-auc:0.75237
[12]	validation_0-auc:0.75317
[13]	validation_0-auc:0.75474
[14]	validation_0-auc:0.75542
[15]	validation_0-auc:0.75681
[16]	validation_0-auc:0.75727
[17]	validation_0-auc:0.75754
[18]	validation_0-auc:0.75837
[19]	validation_0-auc:0.75935
[20]	validation_0-auc:0.76034
[21]	validation_0-auc:0.76034
[22]	validation_0-auc:0.76118
[23]	validation_0-auc:0.76144
[24]	validation_0-auc:0.76141
[25]	validation_0-auc:0.76184
[26]	validation_0-auc:0.76203
[27]	validation_0-auc:0.76263
[28]	validation_0-auc:0.76285
[29]	validation_0-auc:0.76266
[30]	validation_0-auc:0.76304
[31]	validation_0-auc:0.76323
[32]	validation_0-auc:0.76351
[33]	validation_0-au

406